In [ ]:
import numpy as np
from adaptive_latents import datasets
from sim_stim import make_srs
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
import pathlib
import matplotlib.pyplot as plt
from adaptive_latents.plotting_functions import AnimationManager
from adaptive_latents.stim_designer import StimDesigner, OptimizationMethod
from tqdm.autonotebook import tqdm
from IPython.display import display, clear_output



In [ ]:
rng = np.random.default_rng(0)
d = datasets.Zong22Dataset()
data = d.neural_data

srs = make_srs(data, rng, comparison_preset='visualization', n_runs=1, show_tqdm=True)


i= 40
sr = srs['learning from stim'][0]

fig, axs = plt.subplots(ncols=2, figsize=(10,4), sharex=False, sharey=False, layout='constrained')

latents = sr.log['latents'].slice_by_time(slice(30,None))
axs[0].plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
stim_s = sr.log['stim_intended_samples'].t - latents.dt

l = 1
r = 5.1
ax_n = 0
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = axs[ax_n].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs[ax_n].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

for arrow_index in [17, 50]:
    axs[0].annotate('',
                    xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                    xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                    arrowprops=dict(arrowstyle="simple", color='C0'),
                    size=11
                    )


u = sr.stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print(np.linalg.norm(u,ord=0))

high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs[1].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs[1].axvline(stim_t, color='r')


In [ ]:
fig, ax = plt.subplots()


latents = sr.log['latents'].slice_by_time(slice(30,None))
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

i = 40
l = 1
r = 5.1
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')


In [ ]:
stim_s

In [ ]:
colors = {
    'blue':f'#00274C',
    'maize':'#1e7608ff',
    'lqs':'#cccccc',
    'red':'#9A3324',
    'orange':'#D86018',
    'su_blue':'#174992',
}

for i, key in enumerate(list(colors.keys())):
    colors[i] = colors[key]

from matplotlib.animation import FFMpegWriter, PillowWriter, HTMLWriter, ImageMagickWriter
from matplotlib.patches import ArrowStyle

latents = sr.log['latents'].slice_by_time(slice(30,None))


# am = AnimationManager(filename_stem='stim_video', outdir='.', filetype='gif', dpi=400)


fig, axs = plt.subplots(1, 1, figsize=(10,5), squeeze=False, layout='constrained')
fig.set_facecolor(colors['blue'])

"""
         \
       outputfile.mp4
"""

extra_args = '-preset slower -pix_fmt yuv420p -color_range tv -c:v libx264 -crf 18 -colorspace bt709 -color_primaries bt709 -color_trc iec61966-2-1 -movflags faststart'.split(' ') + ['-vf', 'scale=in_color_matrix=bt709:out_color_matrix=bt709']

movie_writer = FFMpegWriter(fps=20, bitrate=-1, extra_args=extra_args)
movie_writer.setup(fig, 'zong_stim.mp4', dpi=400)

# for t in tqdm(np.arange(297, 308.6, 11/201)):
for t in tqdm(np.arange(297, 302.7, 11/201)):
    ax = axs[0,0]
    ax.cla()
    ax.axis('off')
    ax.axis('scaled')
    ax.set_ylim(np.array([-2.5,2.5]) + 0.33)
    ax.set_xlim(np.array([-4,4]) + 1.26)
    ax.add_artist(ax.patch)
    ax.patch.set_zorder(-1)
    ax.set_facecolor(colors['blue'])
    fig.set_facecolor(colors['blue'])


    sl = slice(None,t)
    sub_latents = latents.slice_by_time(sl)
    ax.plot(sub_latents[:, 0], sub_latents[:, 1], alpha=.5, color=colors['lqs'], lw=1.5)

    sl = slice(t-2,t)
    sub_latents = latents.slice_by_time(sl)
    ax.plot(sub_latents[:, 0], sub_latents[:, 1], color='white', lw=3)

    sl = slice(t-2- latents.dt,t- latents.dt)
    stim_s = sr.log['stim_intended_samples'].slice_by_time(sl).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
    ax.scatter(latents_s[:, 0], latents_s[:, 1], s=75, color=colors['maize'], zorder=1000)

    for point in latents_s:
        ax.annotate('',
                    xytext=(point[0], point[1]),
                    xy=(point[0] +1, point[1]),
                    arrowprops=dict(color=colors['maize'], width=1.5),
                    # arrowprops=dict(arrowstyle=ArrowStyle("-|>", head_length=0.4, head_width=0.2), color='red', tail_width=1),
                    size=30
                )
    movie_writer.grab_frame()



ax = axs[0,0]
ax.cla()
ax.axis('off')
ax.axis('scaled')
ax.set_ylim(np.array([-2.5,2.5]) + 0.33)
ax.set_xlim(np.array([-4,4]) + 1.26)
ax.add_artist(ax.patch)
ax.patch.set_zorder(-1)
ax.patch.set_facecolor(colors['blue'])


sl = slice(None,t)
sub_latents = latents.slice_by_time(sl)
ax.plot(sub_latents[:, 0], sub_latents[:, 1], alpha=.5, color=colors['lqs'], lw=1.5)

sl = slice(t-2,t)
sub_latents = latents.slice_by_time(sl)
ax.plot(sub_latents[:, 0], sub_latents[:, 1], color='white', lw=3)
ax.scatter(sub_latents[-1, 0], sub_latents[-1, 1], s=75, color=colors['maize'], zorder=1000)

point = sub_latents[-1]
ax.annotate('',
            xytext=(point[0], point[1]),
            xy=(point[0] +1, point[1]),
            arrowprops=dict(color=colors['orange'], width=1.5),
            # arrowprops=dict(arrowstyle=ArrowStyle("-|>", head_length=0.4, head_width=0.2), color='red', tail_width=1),
            size=30
            )


movie_writer.grab_frame()

movie_writer.finish()

In [ ]:
sub_latents[-1]

In [ ]:
stim_designer = StimDesigner(should_log=True, rng_seed=1)
def plot_frame(axs, sr, i=40, l=1, r=0, opt_method=OptimizationMethod.CHEAT_HIGHD_VEC_MANY_NEURONS, theta=0):
    latents = sr.log['latents'].slice_by_time(slice(30,None))
    ax = axs[1]
    ax.plot(latents[:, 0], latents[:, 1], alpha=.3, color=colors['lqs'])

    center_t = sr.log['stim_intended_samples'].t[i]
    latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
    line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='white')
    stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
    ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color=colors['maize'])



    u = sr.stim_designer.log[i]['u']
    v = sr.stim_designer.log[i]['v']

    v = 0 * v
    v[0,0] = np.cos(theta)
    v[1,0] = np.sin(theta)

    equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
    u_to_s_function=lambda u: equivalent_projection_matrix.T @ u


    stim_designer.optimization_method = opt_method
    new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)

    v_arrow = ax.annotate('',
                          xytext=(latents_s[0,0], latents_s[0,1]),
                          xy=(latents_s[0,0]+v[0,0], latents_s[0,1]+v[1,0]),
                          arrowprops=dict(color=colors['orange'], width=1.5),
                          size=20
                          )

    s = u_to_s_function(new_u)
    s_marker = axs[1].annotate('',
                               xytext=(latents_s[0,0], latents_s[0,1]),
                               xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
                               arrowprops=dict(color=colors['maize'], width=1.5),
                               size=20,
                               ).arrow_patch





    def make_legend_arrow(legend, orig_handle,
                          xdescent, ydescent,
                          width, height, fontsize):
        p = mpatches.FancyArrow(width, 0.5*height, -width, 0, length_includes_head=True, head_width=0.7*height, head_length=.23*width)
        return p

    ax = axs[0]
    ax.matshow(d.ops['meanImg'], cmap='gray')

    c = np.array(new_u)
    xs, ys = list(map(np.array,zip(*[cell['med'] for cell in d.stat])))
    c[c == 0] = np.nan
    # ax.scatter(ys[~np.isnan(c)], xs[~np.isnan(c)], s=20, c='k')
    cmap = ax.scatter(ys, xs, s=20, c=c, vmin=0, vmax=1, cmap='plasma')

    axs[0].axis('off');

    axs[1].axis('equal');
    axs[1].axis('off');
    axs[1].set_xlim(np.array([-2,2]) + 1.26)
    axs[1].set_ylim(np.array([-2,2]) + 0.73)




In [ ]:


fig, axs = plt.subplots(figsize=(10,5), ncols=2, layout='constrained')
fig.set_facecolor(colors['blue'])
movie_writer = FFMpegWriter(fps=15, bitrate=-1)
movie_writer.setup(fig, 'spin_optim.mp4', dpi=400)

for i in tqdm(range(100)):
    for ax in axs.flatten():
        ax.cla()

    plot_frame(axs, sr, theta=i/100*2*np.pi, opt_method=OptimizationMethod.JAXOPT)
    # plot_frame(axs, sr, theta=0, opt_method=OptimizationMethod.CHEAT_HIGHD_VEC_MANY_NEURONS)

    movie_writer.grab_frame()

movie_writer.finish()




In [ ]:
fig, axs = plt.subplots(figsize=(5,5), layout='constrained',squeeze=False)



theta=.1
i=40
l=1
r=0
opt_method=OptimizationMethod.JAXOPT

latents = sr.log['latents'].slice_by_time(slice(30,None))
ax = axs[0,0]
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='C0')
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='g')



u = sr.stim_designer.log[i]['u']
v = sr.stim_designer.log[i]['v']

for theta in np.linspace(0, 2*np.pi, 20) - 0.11:
    v = 0 * v
    v[0,0] = np.cos(theta)
    v[1,0] = np.sin(theta)

    equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
    u_to_s_function=lambda u: equivalent_projection_matrix.T @ u


    stim_designer.optimization_method = opt_method
    new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)

    # v_arrow = ax.annotate('',
    #                       xytext=(latents_s[0,0], latents_s[0,1]),
    #                       xy=(latents_s[0,0]+v[0,0], latents_s[0,1]+v[1,0]),
    #                       arrowprops=dict(color=colors['orange'], width=1.5),
    #                       size=20
    #                       )

    s = u_to_s_function(new_u)*4
    s_marker = axs[0,0].annotate('',
                               xytext=(latents_s[0,0], latents_s[0,1]),
                               xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
                               arrowprops=dict(color=colors['maize'], width=1.5),
                               size=20,
                               ).arrow_patch





    def make_legend_arrow(legend, orig_handle,
                          xdescent, ydescent,
                          width, height, fontsize):
        p = mpatches.FancyArrow(width, 0.5*height, -width, 0, length_includes_head=True, head_width=0.7*height, head_length=.23*width)
        return p


axs[0,0].axis('equal');
axs[0,0].axis('off');
axs[0,0].set_xlim(np.array([-2,2]) + 1.26)
axs[0,0].set_ylim(np.array([-2,2]) + 0.73)



fig.savefig('/home/jgould/Downloads/starburst.svg', dpi=400)
